In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import belo_horizonte_real_estate_market.functions.match_info as match_info
import belo_horizonte_real_estate_market.functions.sources as dicts

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
cols_enderecamento = ['idend', 'id_edc', 'id_logradouro', 'sigla_tipo_logradouro', 'desc_tipo_logradouro', 'nome_logradouro',
                      'numero_imovel', 'letra_imovel', 'cep', 'id_bairro_popular', 'nome_bairro_popular', 'id_bairro_oficial',
                      'tipo_bairro_oficial', 'nome_bairro_oficial', 'nome_regional', 'existencia_num_local', 'situacao_pbh',
                      'geometria']

In [4]:
df_enderecamento = pd.read_parquet(path = "../data/raw_enderecamento.parquet")\
[cols_enderecamento].astype("string").fillna("")\
.pipe(func = match_info.clean_column_text, column = "sigla_tipo_logradouro")\
.pipe(func = match_info.clean_column_text, column = "desc_tipo_logradouro")\
.pipe(func = match_info.clean_column_text, column = "nome_logradouro")\
.pipe(func = match_info.clean_column_text, column = "tipo_bairro_oficial")\
.pipe(func = match_info.clean_column_text, column = "nome_bairro_oficial")\
.pipe(func = match_info.clean_column_text, column = "nome_bairro_popular")\
.pipe(func = match_info.clean_column_text, column = "nome_regional")\
.pipe(func = match_info.clean_column_text, column = "existencia_num_local")\
.pipe(func = match_info.clean_column_text, column = "situacao_pbh")\
.assign(cep = lambda df: df["cep"].str.replace("\.0", "", regex = True))\
.assign(ind_cep = lambda df: df["cep"].apply(lambda x: len(x)))\
.assign(cep = lambda df : np.where(df["ind_cep"] == 8, df["cep"].apply(lambda x: x[:5] + "-" + x[5:]), ""))

In [5]:
df_enderecamento_bhmap = pd.read_parquet(path = "../data/raw_enderecamento_bhmap.parquet")\
[cols_enderecamento].astype("string").fillna("")\
.pipe(func = match_info.clean_column_text, column = "sigla_tipo_logradouro")\
.pipe(func = match_info.clean_column_text, column = "desc_tipo_logradouro")\
.pipe(func = match_info.clean_column_text, column = "nome_logradouro")\
.pipe(func = match_info.clean_column_text, column = "tipo_bairro_oficial")\
.pipe(func = match_info.clean_column_text, column = "nome_bairro_oficial")\
.pipe(func = match_info.clean_column_text, column = "nome_bairro_popular")\
.pipe(func = match_info.clean_column_text, column = "nome_regional")\
.pipe(func = match_info.clean_column_text, column = "existencia_num_local")\
.pipe(func = match_info.clean_column_text, column = "situacao_pbh")\
.assign(cep = lambda df: df["cep"].str.replace("\.0", "", regex = True))\
.assign(ind_cep = lambda df: df["cep"].apply(lambda x: len(x)))\
.assign(cep = lambda df : np.where(df["ind_cep"] == 8, df["cep"].apply(lambda x: x[:5] + "-" + x[5:]), ""))

In [6]:
df_processed_enderecamento = pd.concat(objs = [df_enderecamento_bhmap, df_enderecamento], axis = 0, ignore_index = True)\
.drop_duplicates("idend")

In [7]:
del df_enderecamento, df_enderecamento_bhmap

In [8]:
gdf_zona_homogenea = pd.read_parquet(path = "../data/raw_zona_homogenea_iptu_bhmap.parquet")\
.astype("str")\
.apply(lambda x: x.str.lower())

gdf_zona_homogenea = gpd.GeoDataFrame(
    data = gdf_zona_homogenea[["codigo_zh"]], 
    geometry = gpd.GeoSeries.from_wkt(gdf_zona_homogenea["geometria"]), 
    crs = "EPSG:4326")

In [9]:
gdf_lotes_ctm = pd.read_parquet(path = "../data/raw_lotes_ctm_bhmap.parquet")\
.astype("str")

gdf_lotes_ctm = gpd.GeoDataFrame(
    data = gdf_lotes_ctm, 
    geometry = gpd.GeoSeries.from_wkt(gdf_lotes_ctm["geometria"]), 
    crs = "EPSG:4326")

In [10]:
gdf_enderecamento = gpd.GeoDataFrame(
    data = df_processed_enderecamento[["idend", "geometria"]], 
    geometry = gpd.GeoSeries.from_wkt(df_processed_enderecamento["geometria"].astype(str)), 
    crs = "EPSG:4326")

In [11]:
df_match_enderecamento_zona_homegenea = gpd.sjoin(
    left_df = gdf_enderecamento, 
    right_df = gdf_zona_homogenea, 
    how = "left", predicate = "within")\
[["idend", "codigo_zh"]].drop_duplicates()

In [12]:
df_match_enderecamento_lote_ctm = gpd.sjoin(
    left_df = gdf_enderecamento, 
    right_df = gdf_lotes_ctm, 
    how = "left", predicate = "within")\
[["idend", "nulotctm", "id_quadra_ctm", "area_m2"]]\
.drop_duplicates()

In [13]:
df_processed_enderecamento = df_processed_enderecamento\
.merge(right = df_match_enderecamento_zona_homegenea, how = "left")\
.merge(right = df_match_enderecamento_lote_ctm, how = "left")\
[["idend", "id_edc", "codigo_zh", "nulotctm", "id_quadra_ctm", "area_m2", "id_logradouro", "sigla_tipo_logradouro", "desc_tipo_logradouro", 
  "nome_logradouro", "numero_imovel", 'letra_imovel', "cep", "geometria", "tipo_bairro_oficial", "nome_bairro_oficial", "nome_bairro_popular", 
  "nome_regional", "existencia_num_local", "situacao_pbh"]]


In [14]:
df_processed_enderecamento.to_parquet(path = "../data/processed_enderecamento.parquet", engine = "fastparquet", compression = "zstd")
df_processed_enderecamento

,idend,id_edc,codigo_zh,nulotctm,id_quadra_ctm,area_m2,id_logradouro,sigla_tipo_logradouro,desc_tipo_logradouro,nome_logradouro,numero_imovel,letra_imovel,cep,geometria,tipo_bairro_oficial,nome_bairro_oficial,nome_bairro_popular,nome_regional,existencia_num_local,situacao_pbh
0,12406200267,539435,pa316,140994600150,6308,420.89,124062,rua,rua,castelo de alcobaca,267,,31330-040,POINT (605126.039304405 7801166.85898991),bairro,do castelo,castelo,pampulha,sim,numero oficial
1,02589800275,963552,nt212,180419000650,12059,349.48,25898,rua,rua,jose de paula cotta,275,,31842-080,POINT (612301.407745009 7806273.51170476),bairro,tupi,tupi a,norte,a confimar,numero oficial
2,03209300380,23642,le201,40060700090,1573,431.73,32093,rua,rua,guanhaes,380,,31110-160,POINT (611343.217321981 7798471.34177438),secao suburbana,sexta,colegio batista,leste,sim,numero oficial
3,05620300040,983010,vn313,190095800010,12993,364.91,56203,rua,rua,luzia cirila rodrigues,40,,31520-210,POINT (607773.90993111 7808293.4808703),bairro,sao joao batista,sao joao batista,venda nova,a confimar,numero oficial
4,04399500170,91185,oe211,100222400460,10327,347.95,43995,rua,rua,maria euzebia,170,,30510-360,POINT (605600.945644327 7794356.81622861),bairro,da gameleira,nova gameleira,oeste,sim,numero oficial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
758438,00964100041.,985712,oe101,21057200065,33607,3819.36,9641,rua,rua,bimbarra,41,.,30411-400,POINT (607242.70 7796477.65),bairro,calafate,calafate,oeste,sim,
758439,"01262800972,",985550,oe218,101414700170,21145,220.38,12628,rua,rua,candido de souza,972,",",30510-070,POINT (605398.01 7794157.80),bairro,da gameleira,imbaubas,oeste,sim,
758440,03575700048,986790,no231,NaN,NaN,NaN,35757,rua,rua,itapecerica,48,,31210-030,POINT (610753.42 7797998.85),secao suburbana,sexta,lagoinha,noroeste,a confimar,numero oficial
758441,01433300270A,524968,ba304,110030700255,3006,166.61,14333,rua,rua,professor milton francisco,270,A,30610-200,POINT (604320.49 7793003.72),bairro,das industrias,bairro das industrias i,barreiro,nao,


In [15]:
df_logradouros_bhmap = pd.read_parquet(path = "../data/raw_logradouros_bhmap.parquet")\
.astype("string")

In [16]:
regex_tp_logradouro = "^(ruadepedestre|rua|avenida|alameda|beco|estrada|praca|rodovia|travessa|viadepedestre|viaduto|via|largo|espacolivredeusopublico|trincheira|acesso|tunel)"

df_logradouros = df_processed_enderecamento\
.groupby(["id_logradouro", "desc_tipo_logradouro", "nome_logradouro", "cep", "nome_bairro_popular", "nome_regional", "codigo_zh"], as_index = False, dropna = False).size()\
.assign(logradouro = lambda df: df["desc_tipo_logradouro"] + " " + df["nome_logradouro"])\
.assign(cod1 = lambda df: df["logradouro"].str.replace("a|e|i|o|u|á|ã|é|ê|í|ó|õ|ô|ú|û|ç|\'|\s+", "", regex = True))\
.assign(cod2 = lambda df: df["logradouro"].str.replace("b|c|d|f|g|h|j|k|l|m|n|p|q|r|s|t|v|x|w|y|z|\s+", "", regex = True))\
.assign(cod3 = lambda df: df["logradouro"].str.replace("\'|\s+", "", regex = True))\
.assign(cod3 = lambda df: df["cod3"].str.replace(regex_tp_logradouro, "", regex = True))\
.rename(columns = {"nome_bairro_popular": "bairro"})\
.assign(prefixo_cep = lambda df: df["cep"].apply(lambda x: x[:5]))\
.assign(sufixo_cep = lambda df: df["cep"].apply(lambda x: x[-3:]))\
.assign(logradouro = lambda df: df["logradouro"].replace("\-|\'", " ", regex = True))\
.assign(ind_nome_unico = lambda df: (df.groupby("logradouro")["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cep_unico = lambda df: (df.groupby(["logradouro", "cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_bairro = lambda df: (df.groupby(["logradouro", "bairro"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_regional = lambda df: (df.groupby(["logradouro", "nome_regional"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_zh = lambda df: (df.groupby(["logradouro", "codigo_zh"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod1_unico = lambda df: (df.groupby("cod1")["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod1_unico = lambda df: df["ind_cod1_unico"] * (df["cod1"].apply(lambda x: len(x)) > 4))\
.assign(ind_cod2_unico = lambda df: (df.groupby("cod2")["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod2_unico = lambda df: df["ind_cod2_unico"] * (df["cod2"].apply(lambda x: len(x)) > 4))\
.assign(ind_cod3_unico = lambda df: (df.groupby("cod3")["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_prefixo_cep = lambda df: (df.groupby(["logradouro", "prefixo_cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_sufixo_cep = lambda df: (df.groupby(["logradouro", "sufixo_cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod1_regional = lambda df: (df.groupby(["cod1", "nome_regional"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod1_regional = lambda df: df["ind_cod1_regional"] * (df["cod1"].apply(lambda x: len(x)) > 4))\
.assign(ind_cod2_regional = lambda df: (df.groupby(["cod2", "nome_regional"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod2_regional = lambda df: df["ind_cod2_regional"] * (df["cod2"].apply(lambda x: len(x)) > 4))\
.assign(ind_cod3_regional = lambda df: (df.groupby(["cod3", "nome_regional"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod1_zh = lambda df: (df.groupby(["cod1", "codigo_zh"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod2_zh = lambda df: (df.groupby(["cod2", "codigo_zh"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod3_zh = lambda df: (df.groupby(["cod3", "codigo_zh"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod1_bairro = lambda df: (df.groupby(["cod1", "bairro"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod1_bairro = lambda df: df["ind_cod1_bairro"] * (df["cod1"].apply(lambda x: len(x)) > 4))\
.assign(ind_cod2_bairro = lambda df: (df.groupby(["cod2", "bairro"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod2_bairro = lambda df: df["ind_cod2_bairro"] * (df["cod2"].apply(lambda x: len(x)) > 4))\
.assign(ind_cod3_bairro = lambda df: (df.groupby(["cod3", "bairro"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod1_cep = lambda df: (df.groupby(["cod1", "cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod2_cep = lambda df: (df.groupby(["cod2", "cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod3_cep = lambda df: (df.groupby(["cod3", "cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod1_prefixo_cep = lambda df: (df.groupby(["cod1", "prefixo_cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod2_prefixo_cep = lambda df: (df.groupby(["cod2", "prefixo_cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod3_prefixo_cep = lambda df: (df.groupby(["cod3", "prefixo_cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod1_sufixo_cep = lambda df: (df.groupby(["cod1", "sufixo_cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod2_sufixo_cep = lambda df: (df.groupby(["cod2", "sufixo_cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(ind_cod3_sufixo_cep = lambda df: (df.groupby(["cod3", "sufixo_cep"])["id_logradouro"].transform("nunique") == 1).astype("int"))\
.assign(num_bairros = lambda df: df.groupby("id_logradouro")["bairro"].transform("nunique"))\
.assign(num_regional = lambda df: df.groupby("id_logradouro")["nome_regional"].transform("nunique"))\
.merge(df_logradouros_bhmap[["id_logradouro", "largura_media", "comprimento_logradouro"]], how = "left")\
.assign(soma_ind = lambda df: df[[i for i in df.columns if 'ind_' in i]].sum(axis = 1))\
.sort_values("size", ascending = False)

In [17]:
df_logradouros.to_parquet(path = "../data/processed_logradouros.parquet", engine = "fastparquet", compression = "zstd")
df_logradouros

,id_logradouro,desc_tipo_logradouro,nome_logradouro,cep,bairro,nome_regional,codigo_zh,size,logradouro,cod1,cod2,cod3,prefixo_cep,sufixo_cep,ind_nome_unico,ind_cep_unico,ind_bairro,ind_regional,ind_zh,ind_cod1_unico,ind_cod2_unico,ind_cod3_unico,ind_prefixo_cep,ind_sufixo_cep,ind_cod1_regional,ind_cod2_regional,ind_cod3_regional,ind_cod1_zh,ind_cod2_zh,ind_cod3_zh,ind_cod1_bairro,ind_cod2_bairro,ind_cod3_bairro,ind_cod1_cep,ind_cod2_cep,ind_cod3_cep,ind_cod1_prefixo_cep,ind_cod2_prefixo_cep,ind_cod3_prefixo_cep,ind_cod1_sufixo_cep,ind_cod2_sufixo_cep,ind_cod3_sufixo_cep,num_bairros,num_regional,largura_media,comprimento_logradouro,soma_ind
8719,28133,rua,fernao dias,30285-160,alto vera cruz,leste,le319,565,rua fernao dias,rfrnds,uaeaoia,fernaodias,30285,160,1,1,1,1,1,1,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,4,1,11.64,2856.4,26
7114,19917,rua,desembargador braulio,30285-170,alto vera cruz,leste,le319,547,rua desembargador braulio,rdsmbrgdrbrl,uaeeaaoauio,desembargadorbraulio,30285,170,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,3,1,9.7,2546.85,28
23411,66535,rua,desembargador saraiva,30285-150,alto vera cruz,leste,le319,501,rua desembargador saraiva,rdsmbrgdrsrv,uaeeaaoaaia,desembargadorsaraiva,30285,150,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,9.38,1522.05,28
25567,78241,rua,coletora,30670-050,vila pinho,barreiro,ba227,494,rua coletora,rcltr,uaoeoa,coletora,30670,050,1,1,1,1,1,1,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,1,13.24,1517.08,26
1576,10878,rua,padre argemiro moreira,31995-162,beira-linha,nordeste,ne412,485,rua padre argemiro moreira,rpdrrgmrmrr,uaaeaeiooeia,padreargemiromoreira,31995,162,1,1,1,1,1,1,1,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,14,2,10.46,10357.68,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16646,36734,rua,jacui,31110-050,floresta,leste,le205,1,rua jacui,rjc,uaaui,jacui,31110,050,0,1,1,1,1,0,0,0,1,1,0,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,7,2,16.26,4359.85,22
16635,36690,rua,jacinto olau,31150-430,santa cruz,nordeste,ne110,1,rua jacinto olau,rjcntl,uaaiooau,jacintoolau,31150,430,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,9.98,285.55,28
6,100029,rua,euclides franco,31370-250,braunas,pampulha,pa107,1,rua euclides franco,rcldsfrnc,uaeuieao,euclidesfranco,31370,250,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,3,1,14.97,832.77,28
29322,99753,rua,dona noemi,31530-570,rio branco,venda nova,vn310,1,rua dona noemi,rdnnm,uaoaoei,donanoemi,31530,570,0,1,1,1,1,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,12.42,106.56,24
